# Teardown (Python SDK)

Remove **your own** course resources from the training CDF project. This notebook deletes
everything the **public Cognite SDK** can delete cleanly. Two things it deliberately does
**not** touch — because they need the Toolkit CLI, not the SDK — are covered in
**[Chapter 19](../19-teardown.md)**:

- your three **spaces** (`isp_…`, `ssp_…`) → `cdf data purge space` (interactive confirm)
- your **location filter** (`loc_…`) → `cdf clean --include locations` (the public SDK has no delete for it)

> ⚠️ **Destructive & irreversible.** This runs in **dry-run by default** (`DRY_RUN = True`) —
> it only prints what it *would* delete. Set `DRY_RUN = False` in the first code cell to
> actually delete. Only ever tear down resources whose name carries **your** `YOURNAME`.

In [ ]:
# ---------------------------------------------------------------- setup ----
import os
from pathlib import Path

from cognite.client import CogniteClient, global_config
global_config.disable_pypi_version_check = True
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, OAuthInteractive

# Find the repo root by its markers, so this cell works wherever Jupyter started.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "pyproject.toml").exists() and (p / "training").exists()), HERE)

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        if " #" in v and not v.startswith(('"', "'")):
            v = v.split(" #", 1)[0].rstrip()
        os.environ.setdefault(k, v)      # a real environment variable always wins

missing = [k for k in ("CDF_PROJECT", "CDF_CLUSTER", "IDP_CLIENT_ID")
           if not os.environ.get(k)]
assert not missing, f"Missing {missing}. Copy .env.example to {env_path} and fill it in."


def cdf_client(name: str) -> CogniteClient:
    """Build the client EXPLICITLY.

    `CogniteClient()` with no arguments does not read your .env. The SDK removed
    implicit construction in v8 and raises:
        ValueError: No ClientConfig has been provided
    The branch below is the two-identity rule from Chapter 02, in code.
    """
    base_url = os.environ.get("CDF_URL") or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
    scopes = [s for s in os.environ.get("IDP_SCOPES", f"{base_url}/.default").split(",") if s]

    if os.environ.get("LOGIN_FLOW", "interactive").lower() == "interactive":
        creds = OAuthInteractive(              # you, in a browser -- needs
            authority_url=os.environ["IDP_AUTHORITY_URL"],   # localhost:53000
            client_id=os.environ["IDP_CLIENT_ID"],           # as a redirect URI
            scopes=scopes)
    else:
        creds = OAuthClientCredentials(        # unattended: a service principal
            token_url=os.environ["IDP_TOKEN_URL"],
            client_id=os.environ["IDP_CLIENT_ID"],
            client_secret=os.environ["IDP_CLIENT_SECRET"],
            scopes=scopes)

    return CogniteClient(ClientConfig(
        client_name=name, project=os.environ["CDF_PROJECT"],
        base_url=base_url, credentials=creds))


YOURNAME = os.environ.get("PARTICIPANT", "YOURNAME")   # [CHANGE] if not in .env
client   = cdf_client(f"dm-handson-{YOURNAME}-teardown")

space       = f"isp_{YOURNAME}_TRN"
schema_edm  = f"ssp_{YOURNAME}_TrainingCore_edm"
schema_sdm  = f"ssp_{YOURNAME}_MaintenanceInsight_sdm"
raw_db      = f"rwd_{YOURNAME}_Training_TRN"
model_version = "v1.0.0"


# --- identifiers every chapter uses ---------------------------------------
from cognite.client.data_classes.data_modeling import ViewId
from cognite.client.data_classes import filters as flt
from cognite.client.data_classes.data_modeling.query import (
    Query, QuerySync, NodeResultSetExpression, EdgeResultSetExpression,
    Select, SourceSelector)
from cognite.client.data_classes.data_modeling import (
    NodeId, EdgeId, NodeApply, EdgeApply, NodeOrEdgeData, DirectRelationReference)
from cognite.client.data_classes.raw import Row

from cognite.client.data_classes.aggregations import Count, Avg, Max

INSTANCE_SPACE = space
EDM_SPACE      = schema_edm
SDM_SPACE      = schema_sdm
RAW_DB         = raw_db
MODEL_VERSION  = model_version

ASSET      = ViewId("cdf_cdm", "CogniteAsset",     "v1")
EQUIPMENT  = ViewId("cdf_cdm", "CogniteEquipment", "v1")
ACTIVITY   = ViewId("cdf_cdm", "CogniteActivity",  "v1")
TIMESERIES = ViewId("cdf_cdm", "CogniteTimeSeries","v1")
FILE       = ViewId("cdf_cdm", "CogniteFile",      "v1")
WORKORDER  = ViewId(EDM_SPACE, "WorkOrder",              MODEL_VERSION)
EHP        = ViewId(SDM_SPACE, "EquipmentHealthProfile", MODEL_VERSION)

print("connected:", client.config.project, "| space:", space)
# ---------------------------------------------------------------- teardown ---
DRY_RUN = True          # [CHANGE] True = preview only; False = actually delete


def do(label, fn):
    """Run one deletion. In dry-run it only prints intent; otherwise it runs fn()
    and tolerates an already-absent resource (prints [skip] instead of raising)."""
    if DRY_RUN:
        print(f"[dry-run] would delete: {label}")
        return
    try:
        fn()
        print(f"[deleted]  {label}")
    except Exception as exc:
        print(f"[skip]     {label}  ({type(exc).__name__}: {exc})")


dataset_xid   = f"dts_{YOURNAME}_Training_TRN"
obj_file_xid  = f"file_{YOURNAME}_TRN_3D_21_SEP"
workflow_xid  = f"wkf_{YOURNAME}_Training_TRN"
em_model_xid  = f"em_{YOURNAME}_docs_to_assets"
threed_name   = f"trd_{YOURNAME}_TRN_CAD"

print("DRY_RUN:", DRY_RUN)


In [ ]:
# Every name derives from YOURNAME, mirroring your config / module files.
functions       = [f"fnc_{YOURNAME}_Training_{n}" for n in
                   ("GenerateDatapoints", "DetectDiagramTags", "MatchDocuments",
                    "ParseDatasheet", "Load3DRevision")]
transformations = [f"tra_{YOURNAME}_Training_TRN_Load_{n}" for n in
                   ("Assets", "Equipment", "TimeSeries", "WorkOrders",
                    "WorkOrderOperations")]
workflow_xid = f"wkf_{YOURNAME}_Training_TRN"
raw_db       = f"rwd_{YOURNAME}_Training_TRN"
obj_file_xid = f"file_{YOURNAME}_TRN_3D_21_SEP"   # classic FileMetadata (NOT a DMS instance)
dataset_xid  = f"dts_{YOURNAME}_Training_TRN"
em_model_xid = f"emp_{YOURNAME}_Datasheet_TRN"    # global entity-matching model (only if EM ran)
threed_name  = f"trd_{YOURNAME}_TRN_CAD"          # classic 3D model created by Load3DRevision

print("functions:      ", functions)
print("transformations:", transformations)
print("workflow:", workflow_xid, "| raw db:", raw_db, "| data set:", dataset_xid)
print("obj file:", obj_file_xid, "| EM model:", em_model_xid, "| 3D model:", threed_name)

In [ ]:
# 1) Cognite Functions — global resources, independent of your spaces.
#    (functions.delete has no ignore_unknown_ids, so delete one at a time: a missing
#    one then skips instead of aborting the rest.)
for xid in functions:
    do(f"function {xid}", lambda xid=xid: client.functions.delete(external_id=xid))

# Deleting a Function does NOT delete the source bundle it uploaded. Each deploy leaves
# a classic file `<function externalId>.zip` behind -- invisible in the Functions UI,
# still counted against your files quota. Remove them explicitly.
for xid in functions:
    do(f"function source zip {xid}",
       lambda xid=xid: client.files.delete(external_id=xid))

In [ ]:
# 2) Transformations (Spark SQL jobs). ignore_unknown_ids tolerates already-gone ones.
do(f"{len(transformations)} transformations",
   lambda: client.transformations.delete(external_id=transformations, ignore_unknown_ids=True))

In [ ]:
# 3) Workflow — deleting the workflow also removes its versions.
do(f"workflow {workflow_xid}",
   lambda: client.workflows.delete(external_id=workflow_xid, ignore_unknown_ids=True))

In [ ]:
# 4) RAW database — recursive=True also drops its four tables.
do(f"RAW db {raw_db} (+ tables)",
   lambda: client.raw.databases.delete(name=raw_db, recursive=True))

In [ ]:
# 5) Classic OBJ file — a classic FileMetadata, NOT a DMS instance, so `cdf data purge
#    space` won't remove it. (The two PDFs ARE DMS CogniteFiles and get purged with the
#    instance space in Chapter 19.)
do(f"classic file {obj_file_xid}",
   lambda: client.files.delete(external_id=obj_file_xid))

In [ ]:
# 6) Data set — CDF has NO hard delete for data sets. The most you can do is ARCHIVE
#    (mark metadata + write-protect). May fail if your ACL doesn't allow it — that's an
#    acceptable end state (the data set simply stays, ideally archived).
from cognite.client.data_classes import DataSetUpdate

def _archive_dataset():
    update = (DataSetUpdate(external_id=dataset_xid)
              .metadata.add({"archived": "true"})
              .write_protected.set(True))
    client.data_sets.update(update)

do(f"archive data set {dataset_xid}", _archive_dataset)

In [ ]:
# 7) Entity-matching model — GLOBAL to the project (not scoped to your space), so it
#    MUST be cleaned up. With the regex-first MatchDocuments cascade a normal run creates
#    no model (em_ran: false); this only matters if you forced EM in the notebook/Function.
do(f"entity-matching model {em_model_xid}",
   lambda: client.entity_matching.delete(external_id=em_model_xid))

In [ ]:
# 8) Classic 3D model created by Load3DRevision via the /3d/models API — NOT a DMS
#    instance, so the space purge won't touch it. Find it by name, then delete by id
#    (deleting the model cascades its revisions).
def _delete_3d_model():
    mine = [m for m in client.three_d.models.list(limit=None) if m.name == threed_name]
    if not mine:
        raise LookupError(f"no 3D model named {threed_name}")
    for m in mine:
        client.three_d.models.delete(id=m.id)

do(f"classic 3D model {threed_name}", _delete_3d_model)

## What this notebook does NOT delete — finish in the terminal

Two resource classes need the Toolkit CLI, not the SDK — see **[Chapter 19](../19-teardown.md)**:

1. **Your three spaces** and all their DMS instances (assets, equipment, time series +
   datapoints, the two PDF `CogniteFile`s + content, diagram-annotation edges, the EHP
   node): `cdf data purge space …` — **instance space first**, interactive confirm.
2. **Your location filter** `loc_<YOURNAME>_TRN`: `cdf clean --include locations`.

Run the verify cell below **after** you've also run the Chapter 19 commands.

In [ ]:
# Verify your global resources are gone (run after the Chapter 19 space purge too).
mine = lambda xs: [x.external_id for x in xs if x.external_id and YOURNAME in x.external_id]
print("your functions left:      ", mine(client.functions.list(limit=None)))
print("your transformations left:", mine(client.transformations.list(limit=None)))
print("your 3D models left:      ",
      [m.name for m in client.three_d.models.list(limit=None) if m.name and YOURNAME in m.name])
try:
    ds = client.data_sets.retrieve(external_id=dataset_xid)
    state = "archived" if ds and (ds.metadata or {}).get("archived") == "true" else "still active"
    print(f"data set {dataset_xid}: {state} (data sets are never hard-deleted)")
except Exception as exc:
    print("data set retrieve:", exc)